In [9]:
import os, json, re
from pathlib import Path
from openai import OpenAI
from datetime import datetime

KEY_FILE = r"E:\Research_projects\ACC\transcribe_experiment\prefix_experiment\key.txt"
os.environ['OPENAI_API_KEY'] = Path(KEY_FILE).read_text().strip()

# Minimal st stub so _call_llm_json can log
class _State(dict):
    def __getattr__(self, k): return self.get(k)
    def __setattr__(self, k, v): self[k] = v

class _St:
    session_state = _State({'agent_logs': []})
st = _St()

# Load constants, prompts, and agent functions from streamlit_app.py (stop before Drive/UI code)
src = Path('streamlit_app.py').read_text(encoding='utf-8')
stop = src.index('# Google Drive helpers')
ns = {'st': st, 'os': os, 'json': json, 're': re, 'datetime': datetime, 'OpenAI': OpenAI}
ns['_openai_client'] = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
exec(src[src.index('MODEL'):stop], ns)
globals().update(ns)

print('Loaded. Questions:', [q['id'] for q in QUESTIONS])

# Interview state
chat = []
q_idx = 0

def run_turn(skip_dm=False):
    global q_idx, chat
    chat_str = _format_chat_for_prompt(chat)
    current_q = QUESTIONS[q_idx]['main_question']
    current_probes = _get_probes_str(q_idx)

    decision = None
    if not skip_dm:
        decision = _run_decision_maker(chat_str, current_q, current_probes)
        action = decision.get('decision', 'FOLLOW_UP')
        print(f"DM: {action} -- {decision.get('reason_for_decision', '')[:120]}")
        if action == 'MOVE_NEXT':
            q_idx += 1
            if q_idx >= len(QUESTIONS):
                print('Interview ended.')
                return None
            current_q = QUESTIONS[q_idx]['main_question']
            current_probes = _get_probes_str(q_idx)
        elif action == 'END_INTERVIEW':
            print('Interview ended.')
            return None

    result = _run_question_generator(chat_str, current_q, decision, current_probes)
    result['question_id'] = QUESTIONS[q_idx]['id']

    chat.append({'role': 'assistant', 'content': result['question_text'], 'options': result.get('options', [])})

    print(f"\n[{result['question_id']}] {result['question_text']}")
    for opt in result.get('options', []):
        print(f"  - {opt['label']}")
    return result

# Opening question -- skip DM
run_turn(skip_dm=True)

Loaded. Questions: ['A1', 'A2', 'B1', 'B2', 'B3', 'B4', 'C1', 'C2', 'C3']

[A1] When someone does not understand you, what do you usually do?
  - Repeat
  - Rephrase
  - Write it down / Type
  - Use a communication device
  - Ask to slow down
  - Gesture
  - Show written version
  - Other / type your answer
  - Skip
  - I'm not sure
  - Prefer not to answer


{'question_id': 'A1',
 'question_text': 'When someone does not understand you, what do you usually do?',
 'answer_mode': 'multiple_choice',
 'options': [{'label': 'Repeat'},
  {'label': 'Rephrase'},
  {'label': 'Write it down / Type'},
  {'label': 'Use a communication device'},
  {'label': 'Ask to slow down'},
  {'label': 'Gesture'},
  {'label': 'Show written version'},
  {'label': 'Other / type your answer'},
  {'label': 'Skip'},
  {'label': "I'm not sure"},
  {'label': 'Prefer not to answer'}],
 'participant_instruction': 'You can choose one option or type your own answer.',
 'why_this_question': 'This will reveal the first repair strategies you use when someone doesn’t understand your speech.',
 'target_information_gap': 'What strategies do speakers with dysarthria use to repair communication when misunderstood, and how often are they used?'}

In [10]:
USER_MESSAGE = "Repeat myself; Rephrase what I said"  # change this

chat.append({'role': 'user', 'content': USER_MESSAGE})
run_turn()

DM: FOLLOW_UP -- The participant identified 'Repeat myself' as a strategy. A single, focused follow-up on when they choose to repeat (vs.

[A1] You mentioned repeating what you said. What factor makes you choose to repeat exactly instead of rephrasing when someone doesn't understand you?
  - Exact words matter
  - Time pressure
  - Preserve emphasis
  - Can't think of a quick rephrase
  - To keep meaning clear
  - Habit / automatic
  - Other / type your answer
  - Skip


{'question_id': 'A1',
 'question_text': "You mentioned repeating what you said. What factor makes you choose to repeat exactly instead of rephrasing when someone doesn't understand you?",
 'answer_mode': 'multiple_choice',
 'options': [{'label': 'Exact words matter'},
  {'label': 'Time pressure'},
  {'label': 'Preserve emphasis'},
  {'label': "Can't think of a quick rephrase"},
  {'label': 'To keep meaning clear'},
  {'label': 'Habit / automatic'},
  {'label': 'Other / type your answer'},
  {'label': 'Skip'}],
 'participant_instruction': 'You can choose one option or type your own answer.',
 'why_this_question': 'This follow-up asks what specifically triggers choosing to repeat over rephrasing, based on your experience.',
 'target_information_gap': "What specific factor triggers your choice to repeat exactly what you said (instead of rephrasing) when someone doesn't understand you?"}

In [ ]:
current = QUESTIONS[q_idx]['id'] if q_idx < len(QUESTIONS) else 'END'
print(f'Current question: {q_idx} ({current})')
print()
for msg in chat:
    role = 'Interviewer' if msg['role'] == 'assistant' else 'Participant'
    print(f"{role}: {msg['content']}")
    if msg.get('options'):
        print(f"  Options: {[o['label'] for o in msg['options']]}")